# Demo for Loading Solver Design Points.

This script is intended to help with loading a design of solver factors to run on one or more problems.

If more than one problem are used a cross-design of problems will be run.

The design file should be a txt or csv file with headers as
factor names and each row representing an individual design point.

Design files generated by the simopt GUI can also be run using this script.

Outputs are printed to the experiments folder in simopt.

## Append SimOpt Path

Since the notebook is stored in simopt/notebooks, we need to append the parent simopt directory to the system path to import the necessary modules later on.

In [ ]:
import sys
from pathlib import Path

# Take the current directory, find the parent, and add it to the system path
sys.path.append(str(Path.cwd().parent))

## Configuration Parameters

This section defines the core parameters for the demo.

To query model/problem/solver names, run `python scripts/list_directories.py`

In [ ]:
from pathlib import Path

# run this script in the terminal from the simopt directory
# name of solver that design was created on
solver_name = "RNDSRCH"
# list of problem names for solver design to be run on
# (if more than one version of same problem, repeat name)
problem_names = [
    "SSCONT-1",
    "SAN-1",
]

# name of file containing design points (csv or excel): column headers must exactly
# match names of solver factors w/ each row representing a design point
# (can also use csv's generated by GUI)

design_filename = Path("load_solver_design_RNDSRCH_design.csv")

# list of dictionaries that provide fixed factors for problems when you don't want
# to use the default values, if you want to use all default values use empty
# dictionary, order must match problem names
problem_fixed_factors = [
    {"budget": 2000, "demand_mean": 90.0, "fixed_cost": 25},
    {"budget": 500},
]

# uncomment this version to run w/ only default problem factors
# problem_fixed_factors = [{},{}]

# use this dictionary to change any default solver factors that were not included
# in the design
solver_fixed_factors = {}

# number of macroreplication to run and each solver design point
n_macroreps = 2
# number of post replications to run on each macro replication
n_postreps = 100
# number of normalization postreplications to run at initial solution and optimal
# solution
n_postreps_init_opt = 200

In [ ]:
# turn design file into df & retrive dp information
import pandas as pd

design_table = pd.read_csv(design_filename)

design_factor_names = design_table.columns.tolist()

# remove GUI columns from list if present
gui_columns = [
    "design_num", "name", "design_type", "num_stacks"
]
for gui_col in gui_columns:
    if gui_col in design_factor_names:
        design_factor_names.remove(gui_col)

dp_list = []  # list of all design points

for _, row in design_table.iterrows():
    dp = {}  # dictionary of current dp
    for factor in design_factor_names:
        dp[factor] = row[factor]
    dp_list.append(dp)

# add fixed solver factors to dps
for fixed_factor in solver_fixed_factors:
    for dp in dp_list:
        dp[fixed_factor] = solver_fixed_factors[fixed_factor]

In [ ]:
from simopt.directory import problem_directory, solver_directory
from simopt.experiment.api import SimulationConfig, run
from simopt.experiment.single import ProblemSolver
from simopt.options import CrnOptions
from simopt.problem import Problem

def make_problem(name: str, factors: dict) -> Problem:
    """Split design-point factors into problem and model factors."""
    default = problem_directory[name]()
    return problem_directory[name](
        fixed_factors={k: v for k, v in factors.items() if k in default.specifications},
        model_fixed_factors={
            k: v for k, v in factors.items() if k in default.model.specifications
        },
    )

solvers = [solver_directory[solver_name](fixed_factors=dp) for dp in dp_list]
problems = [
    make_problem(name, factors)
    for name, factors in zip(problem_names, problem_fixed_factors, strict=True)
]

experiments = [
    ProblemSolver(solver=solver, problem=problem) for problem in problems for solver in solvers
]

In [ ]:
simulation_config = SimulationConfig(
    n_mreps=n_macroreps,
    n_preps=n_postreps,
    n_preps_x0_xstar=n_postreps_init_opt,
)

In [ ]:
# Run macroreplications, postreplications and normalization at each design point.
analysis_inputs = run(experiments, simulation_config)

In [ ]:
# Inspect the results of the first problem-solver pair.
analysis_inputs[0].mean_df.head()